# 第11周 · Attention 与 Transformer

**本周结束时你能做到**：说清 token 与 embedding（`nn.Embedding` 就是查表）；把注意力理解为"按相似度加权平均"，并在纸上算一次；实现缩放点积注意力、因果掩码与 padding 掩码、多头注意力；解释残差连接、LayerNorm 与位置编码各管什么；用自己写的模块搭出 Transformer 编码器，并亲眼看到"没有位置编码就分不清顺序"。

本周的实现蓝本是 Phillip Lippe 的 **UvA Deep Learning Tutorial 6** [Transformers and Multi-Head Attention](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial6/Transformers_and_MHAttention.html)（仓库 [phlippe/uvadlc_notebooks](https://github.com/phlippe/uvadlc_notebooks)，MIT 许可）：`scaled_dot_product`、`MultiheadAttention`、`EncoderBlock`、`PositionalEncoding` 的代码结构照它来写，S4 的序列反转任务也是它的原题。我们把它用到的 PyTorch Lightning 去掉，改用第 8 周你亲手写的训练循环。原始论文是 Vaswani et al. 2017, [Attention Is All You Need](https://arxiv.org/abs/1706.03762)。

**本周规矩**：每一行张量运算旁边用注释写出 shape，例如 `# (B, T, C)`。B = batch 大小，T = 序列长度（token 数），C = 每个 token 向量的维度。`(B, T, C)` 三维张量的维度管理是本周头号难点——写注释不是形式主义，是不迷路的唯一办法。（选学工具：[einops](https://github.com/arogozhnikov/einops) 能把 reshape/permute 写成 `"b t (h d) -> b h t d"` 这样自带说明的字符串；本课不依赖它。）

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import math
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from zero2researcher import experiments
from zero2researcher.checks import week11 as ck

# 课程统一的 device 选择：有 CUDA 用 CUDA（Colab GPU），Apple 芯片用 MPS，否则 CPU
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

---
## S1 · token、embedding 与"按相似度加权平均"

### nn.Embedding 就是查表

第 10 周的 TextCNN 第一层是 `nn.Embedding`：每个 token（词或字符）先变成一个整数 id，再换成一个可学习的向量。它内部就是一张 `(词表大小, 维度)` 的表，第 $i$ 个 token 取第 $i$ 行。

数学上，"取第 $i$ 行"等价于"one-hot 向量乘以这张表"：one-hot 只有第 $i$ 位是 1，乘矩阵恰好把第 $i$ 行挑出来。验证一下：

In [ ]:
torch.manual_seed(0)
emb = nn.Embedding(num_embeddings=10, embedding_dim=4)   # 词表 10 个 token，每个 token 一个 4 维向量
ids = torch.tensor([3, 1, 3])                             # (T,) = (3,)：一句话的 3 个 token id

# 在这里写你的代码
raise NotImplementedError
print("查表结果:\n", looked_up.detach())
print("one-hot 乘矩阵 与 查表 的最大差异:", (looked_up - via_matmul).abs().max().item())

差异是 0：**embedding 层 = 输入为 one-hot 的线性层（无偏置）**。查表只是省掉了一次和大量 0 相乘的矩阵乘法——词表有 2 万个词时，这个省法很可观。（UvA 教程的序列反转模型正是"one-hot + `nn.Linear`"，我们用 `nn.Embedding`，两者等价。）

### 注意力：按相似度加权平均

一句话里的每个 token，都想从其他 token 那里收集信息。注意力给每个 token $i$ 准备三个向量：

- **query** $q_i$：我在找什么；
- **key** $k_j$：每个 token 贴在自己身上的"标签"；
- **value** $v_j$：每个 token 真正交出去的内容。

token $i$ 的新表示 = 所有 value 的**加权平均**，权重由 $q_i$ 与各个 $k_j$ 的**相似度**（点积）决定：

$$\text{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

除以 $\sqrt{d_k}$ 是为了数值稳定（S2 专门讲）。先用 3 个 token、$d_k = 4$（所以 $\sqrt{d_k} = 2$）在纸上算一次：

$$Q = \begin{bmatrix} 1&0&1&0 \\ 0&2&0&2 \\ 1&1&1&1 \end{bmatrix},\quad K = \begin{bmatrix} 1&0&1&0 \\ 0&1&0&1 \\ 1&1&0&0 \end{bmatrix},\quad V = \begin{bmatrix} 1&0 \\ 0&1 \\ 1&1 \end{bmatrix}$$

**token 1**（$q_1 = [1, 0, 1, 0]$）的计算过程：

| $j$ | $k_j$ | $q_1 \cdot k_j$ | $\div \sqrt{4}$ | $e^{(\cdot)}$ | 权重 $w_{1j}$ | $v_j$ |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | $[1,0,1,0]$ | 2 | 1.0 | 2.718 | 0.506 | $[1, 0]$ |
| 2 | $[0,1,0,1]$ | 0 | 0.0 | 1.000 | 0.186 | $[0, 1]$ |
| 3 | $[1,1,0,0]$ | 1 | 0.5 | 1.649 | 0.307 | $[1, 1]$ |
| | | | | 合计 5.367 | 合计 1 | |

权重 = $e^{(\cdot)}$ 除以合计（这就是 softmax）。输出 = 加权平均：

$$0.506 \cdot [1, 0] + 0.186 \cdot [0, 1] + 0.307 \cdot [1, 1] = [0.814,\ 0.494]$$

token 1 和 token 1 自己最像（点积 2），所以它的新表示里 $v_1$ 的份额最大。用 NumPy 把三个 token 一起算出来：

In [ ]:
Q = np.array([[1, 0, 1, 0], [0, 2, 0, 2], [1, 1, 1, 1]], dtype=float)   # (T, d_k) = (3, 4)
K = np.array([[1, 0, 1, 0], [0, 1, 0, 1], [1, 1, 0, 0]], dtype=float)   # (T, d_k) = (3, 4)
V = np.array([[1, 0], [0, 1], [1, 1]], dtype=float)                     # (T, d_v) = (3, 2)

# 在这里写你的代码
raise NotImplementedError
print("权重矩阵（第 i 行 = token i 分给各个 token 的权重）:\n", weights.round(3))
print("输出（每行是一个 token 的新表示）:\n", out.round(3))

第一行正是表格里手算的 $[0.506, 0.186, 0.307]$ 与 $[0.814, 0.494]$。注意第 3 行：$q_3 = [1,1,1,1]$ 与三个 key 的点积都是 2，于是权重均分、输出就是三个 value 的平均。

### 实现 scaled_dot_product

照 UvA 教程把它写成 PyTorch 函数：输入可以带任意多个前置维度（`(B, T, d)` 或 `(B, H, T, d)`），所以转置只转**最后两维**、softmax 沿**最后一维**。函数同时返回输出和注意力权重（后面画图要用）。`mask` 参数 S2 再讲，先照写：

In [ ]:
def scaled_dot_product(q, k, v, mask=None):
    """q, k: (..., T, d_k)；v: (..., T, d_v)；mask 可广播到 (..., T, T)，1 = 保留、0 = 遮住。
    返回 (values, attention)。结构照 UvA Tutorial 6（MIT）。"""
    # 在这里写你的代码
    raise NotImplementedError
    return values, attention


values, attn = scaled_dot_product(torch.tensor(Q), torch.tensor(K), torch.tensor(V))
print("PyTorch 版 token 1 的输出:", values[0].numpy().round(3), "  （手算 [0.814, 0.494]）")
ck.scaled_dot_product(scaled_dot_product)

自查不只和课程的参考实现比，还和 PyTorch 内置的 `F.scaled_dot_product_attention` 比——又一道"阳性对照"。

本周最常见的报错长什么样？故意忘掉转置，让矩阵乘法的内维对不上，读一读报错信息里的两个 shape：

In [ ]:
q_demo = torch.randn(2, 5, 4)      # (B, T, d)
q_demo @ q_demo                    # (2, 5, 4) @ (2, 5, 4)：前者的最后一维 4 ≠ 后者的倒数第二维 5

报错说：第二个张量（`batch2`）的前两维应该是 `[2, 4]`（batch 大小 2、内维 4），实际却是 `[2, 5]`。翻译成人话：矩阵乘法 `A @ B` 要求 A 的最后一维等于 B 的倒数第二维，这里 4 ≠ 5。改成 `q_demo @ q_demo.transpose(-2, -1)` 就是 `(2, 5, 4) @ (2, 4, 5) → (2, 5, 5)`。以后遇到 shape 报错，第一件事就是把参与运算的每个张量的 shape 打印出来，对照这条规则。

---
## S2 · 为什么除以 √d、两种掩码、多头注意力

### 为什么要除以 $\sqrt{d_k}$

Vaswani et al. 2017 §3.2.1 的脚注给了一个简洁的论证：如果 $q$ 和 $k$ 的各个分量相互独立、均值 0、方差 1，那么点积 $q \cdot k = \sum_{i=1}^{d_k} q_i k_i$ 是 $d_k$ 个"均值 0、方差 1"的项之和——**均值 0、方差 $d_k$**，标准差 $\sqrt{d_k}$。

$d_k = 256$ 时，未缩放的点积标准差是 16：几个分数之间动辄差出几十，softmax 几乎把全部权重压到一个 key 上（**饱和**），而饱和的 softmax 梯度接近 0，模型学不动。除以 $\sqrt{d_k}$ 正好把方差拉回 1。数值验证：

In [ ]:
torch.manual_seed(0)
d = 256
q = torch.randn(10000, d)          # (N, d)：1 万个随机 query，分量 ~ N(0, 1)
k = torch.randn(10000, d)          # (N, d)

# 在这里写你的代码
raise NotImplementedError
print(f"未缩放点积的标准差: {dots.std():.2f}   （理论值 √{d} = {math.sqrt(d):.0f}）")
print(f"除以 √d 之后的标准差: {(dots / math.sqrt(d)).std():.2f}")
ck.unscaled_std(dots.std())

In [ ]:
logits = q[0] @ k[:8].T                                  # (8,)：一个 query 对 8 个 key 的未缩放分数
p_raw = F.softmax(logits, dim=-1)                         # (8,)
p_scaled = F.softmax(logits / math.sqrt(d), dim=-1)       # (8,)


def softmax_grad_norm(p):
    """softmax 的雅可比矩阵 diag(p) − p pᵀ 的范数：权重接近 one-hot 时它接近 0（梯度消失）"""
    return (torch.diag(p) - torch.outer(p, p)).norm().item()


print(f"未缩放：最大权重 {p_raw.max():.4f}   梯度范数 {softmax_grad_norm(p_raw):.2e}")
print(f"缩放后：最大权重 {p_scaled.max():.4f}   梯度范数 {softmax_grad_norm(p_scaled):.2e}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
axes[0].bar(range(8), p_raw)
axes[0].set_title("Unscaled: softmax collapses onto one key")
axes[1].bar(range(8), p_scaled, color="C1")
axes[1].set_title("Divided by sqrt(d): weights stay spread")
for ax in axes:
    ax.set_xlabel("Key index")
axes[0].set_ylabel("Attention weight")
plt.tight_layout()
plt.show()

### 两种掩码

注意力默认"人人看得见人人"。两种场合必须遮住一部分：

1. **因果掩码（causal mask）**：生成文本时，第 $i$ 个位置只能看自己和之前的位置——否则预测下一个字时等于偷看了答案。掩码是下三角矩阵 `torch.tril(torch.ones(T, T))`，第 $i$ 行只保留 $j \le i$。下周的 GPT 全靠它。
2. **padding 掩码**：一个 batch 里的句子长短不一，短的要在末尾补 `<pad>`（id = 0，`text.imdb_arrays` 就是这么做的）。真实 token 不应该把注意力分给 `<pad>`。掩码形状取 `(B, 1, 1, T)`：对每个句子遮住 pad 所在的**列**（key），自动广播到所有头、所有 query。

实现上两者是同一招：在 softmax **之前**把要遮住的分数填成 $-\infty$，$e^{-\infty} = 0$，权重自然为 0，且剩下的权重仍然归一化。

In [ ]:
T = 5
causal = torch.tril(torch.ones(T, T, dtype=torch.bool))       # (T, T)：第 i 行只保留 j ≤ i
print(causal.int())

ids = torch.tensor([[5, 8, 2, 9, 4],
                    [7, 3, 6, 0, 0]])                          # (B, T) = (2, 5)：第 2 句末尾两个是 <pad> = 0
# 在这里写你的代码
raise NotImplementedError
print("padding 掩码的 shape:", tuple(pad_mask.shape))

torch.manual_seed(0)
x = torch.randn(2, 1, T, 4)                                    # (B, H, T, d)：2 句、1 个头
_, attn_none = scaled_dot_product(x, x, x)                     # (2, 1, 5, 5)
_, attn_causal = scaled_dot_product(x, x, x, mask=causal)
_, attn_pad = scaled_dot_product(x, x, x, mask=pad_mask)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
panels = [(attn_none, "No mask: everyone sees everyone"),
          (attn_causal, "Causal mask: only the past"),
          (attn_pad, "Padding mask: pads get zero weight")]
for ax, (a, title) in zip(axes, panels):
    ax.imshow(a[1, 0], cmap="viridis", vmin=0, vmax=1)        # 画第 2 句（带 pad 的那句）
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Key position")
    ax.set_ylabel("Query position")
plt.tight_layout()
plt.show()

### 多头注意力

一组 $(Q, K, V)$ 只能表达一种"相似"。**多头**就是并行地做 $H$ 组注意力，每个头有自己的投影、关注不同的关系（比如一个头盯语法、一个头盯指代），各自在 $d = C / H$ 维里算，最后拼回 $C$ 维、再过一个输出投影。

UvA 的实现有个省事的技巧：用**一个** `nn.Linear(C, 3C)`（`qkv_proj`）一次性算出所有头的 $q, k, v$，再用 reshape/permute 把头的维度拆出来。每一步的 shape（B = 2、T = 5、C = 16、H = 4、d = 4 时）：

| 步骤 | 代码 | shape |
| --- | --- | --- |
| 输入 | `x` | (B, T, C) = (2, 5, 16) |
| 合并投影 | `qkv_proj(x)` | (B, T, 3C) = (2, 5, 48) |
| 拆出头 | `.reshape(B, T, H, 3d)` | (2, 5, 4, 12) |
| 头放到前面 | `.permute(0, 2, 1, 3)` | (B, H, T, 3d) = (2, 4, 5, 12) |
| 切成三份 | `.chunk(3, dim=-1)` | q、k、v 各 (2, 4, 5, 4) |
| 每个头各算注意力 | `scaled_dot_product` | 输出 (2, 4, 5, 4)，权重 (B, H, T, T) = (2, 4, 5, 5) |
| 头放回去、拼接 | `.permute(0, 2, 1, 3).reshape(B, T, C)` | (2, 5, 16) |
| 输出投影 | `o_proj` | (2, 5, 16) |

In [ ]:
class MultiheadAttention(nn.Module):
    """多头自注意力（UvA Tutorial 6 的结构；构造参数简化为 embed_dim, num_heads）"""

    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim 必须能被 num_heads 整除"
        self.embed_dim, self.num_heads = embed_dim, num_heads
        self.head_dim = embed_dim // num_heads
        self.qkv_proj = nn.Linear(embed_dim, 3 * embed_dim)   # 一次算出所有头的 q、k、v
        self.o_proj = nn.Linear(embed_dim, embed_dim)
        nn.init.xavier_uniform_(self.qkv_proj.weight)          # 初始化照 UvA（原 Transformer 论文的做法）
        self.qkv_proj.bias.data.fill_(0)
        nn.init.xavier_uniform_(self.o_proj.weight)
        self.o_proj.bias.data.fill_(0)

    def forward(self, x, mask=None, return_attention=False):
        B, T, C = x.shape
        # 在这里写你的代码
        raise NotImplementedError
        return (o, attention) if return_attention else o


mha = MultiheadAttention(embed_dim=16, num_heads=4)
x = torch.randn(2, 5, 16)                                          # (B, T, C)
out, attn = mha(x, mask=causal, return_attention=True)
print("输出:", tuple(out.shape), " 注意力权重:", tuple(attn.shape))
ck.multihead_shapes(MultiheadAttention)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for h, ax in enumerate(axes):
    ax.imshow(attn[0, h].detach(), cmap="viridis", vmin=0, vmax=1)
    ax.set_title(f"Head {h}")
    ax.set_xlabel("Key position")
axes[0].set_ylabel("Query position")
fig.suptitle("Each head computes its own weights (untrained, causal mask)")
plt.tight_layout()
plt.show()

---
## S3 · Transformer 块与位置编码

### EncoderBlock：注意力 + MLP + 残差 + LayerNorm

一个 Transformer 编码器块（UvA 与原论文的 **Post-LN** 写法）：

$$x \leftarrow \mathrm{LayerNorm}\big(x + \mathrm{MHA}(x)\big), \qquad x \leftarrow \mathrm{LayerNorm}\big(x + \mathrm{FFN}(x)\big)$$

- **多头注意力**负责"交流"：token 之间交换信息；
- **FFN**（两层 MLP，中间维度通常是 $C$ 的 2–4 倍）负责"思考"：对每个 token **各自**做非线性变换；
- **残差连接** $x + f(x)$：每层只需学一个"修正量"，梯度也能沿着加法这条高速路直通底层——深网络能训起来的关键（第 9 周梯度消失的解药之一）；
- **LayerNorm**：把每个 token 的 $C$ 维向量各自标准化（均值 0、方差 1），再乘可学习的缩放 $\gamma$、加可学习的平移 $\beta$，让每层的输入尺度稳定。

（下周 Karpathy 的 GPT 用的是 **Pre-LN**：`x = x + attn(ln(x))`，先归一化再进子层——更容易训练，现代大模型基本都这么写。两种写法都对，读代码时认清就好。）

In [ ]:
class EncoderBlock(nn.Module):
    """Transformer 编码器块（UvA Tutorial 6 的结构，Post-LN）"""

    def __init__(self, input_dim, num_heads, dim_feedforward, dropout=0.0):
        super().__init__()
        self.self_attn = MultiheadAttention(input_dim, num_heads)
        self.linear_net = nn.Sequential(                    # FFN：C → d_ff → C
            nn.Linear(input_dim, dim_feedforward),
            nn.Dropout(dropout),
            nn.ReLU(inplace=True),
            nn.Linear(dim_feedforward, input_dim),
        )
        self.norm1 = nn.LayerNorm(input_dim)
        self.norm2 = nn.LayerNorm(input_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # 在这里写你的代码
        raise NotImplementedError
        return x


block = EncoderBlock(input_dim=16, num_heads=4, dim_feedforward=32)
print("输入 (2, 5, 16) → 输出", tuple(block(torch.randn(2, 5, 16)).shape))

### 位置编码的必要性：现场打乱顺序

注意力是对一**组** value 做加权平均，权重只看内容相似度——它根本不知道谁在前、谁在后。FFN 和 LayerNorm 又是对每个 token 各自做的。所以整个块是**置换等变**的：把输入的 token 顺序打乱，输出只是按同样的方式被打乱，每个 token 得到的新向量一模一样。

现场验证。先打乱、再过块，与"先过块、再按同样的顺序打乱输出"比较：

In [ ]:
experiments.set_seed(0)
block = EncoderBlock(input_dim=16, num_heads=4, dim_feedforward=32).eval()   # eval：关掉 dropout
x = torch.randn(1, 6, 16)                     # (B, T, C)：一句 6 个 token 的话
perm = torch.randperm(6)                      # 一个随机的新顺序
print("打乱后的顺序:", perm.tolist())

with torch.no_grad():
    # 在这里写你的代码
    raise NotImplementedError
print(f"逐 token：max |f(x[perm]) − f(x)[perm]| = {diff_tokens:.2e}")
print(f"mean-pool 之后的句子表示：最大差异 = {diff_pooled:.2e}")

两个差异都只是浮点误差。第二个数更扎心：如果分类器只是"编码器 + 对 token 取平均"，那么 *the dog bit the man* 和 *the man bit the dog* 会得到**完全相同**的句子表示。语序信息必须从外面注入——这就是**位置编码**。

### 正弦位置编码

原论文给每个位置 $pos$ 造一个 $C$ 维的"指纹"，直接**加**到 token embedding 上：

$$PE_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{2i/C}}\right), \qquad PE_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/C}}\right)$$

低维分量频率高（相邻位置变化快），高维分量频率低（远距离才变）——像一个多指针的时钟，每个位置的读数都不同。实现照 UvA：

In [ ]:
class PositionalEncoding(nn.Module):
    """正弦位置编码（Vaswani et al. 2017；代码结构照 UvA Tutorial 6）"""

    def __init__(self, d_model, max_len=5000):
        super().__init__()
        # 在这里写你的代码
        raise NotImplementedError
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)             # (1, max_len, C)：不参与训练

    def forward(self, x):                                                         # x: (B, T, C)
        return x + self.pe[:, : x.size(1)]                                        # 广播加到每个句子上


pe_layer = PositionalEncoding(d_model=48, max_len=96)
pe = pe_layer.pe[0]                                                                # (96, 48)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
im = axes[0].imshow(pe.T, cmap="RdBu", aspect="auto")
axes[0].set_xlabel("Position")
axes[0].set_ylabel("Dimension")
axes[0].set_title("Each position gets a unique sinusoidal fingerprint")
fig.colorbar(im, ax=axes[0])
for dim in [0, 6, 20]:
    axes[1].plot(pe[:, dim], label=f"dim {dim}")
axes[1].set_xlabel("Position")
axes[1].set_ylabel("PE value")
axes[1].set_title("Low dimensions oscillate fast, high ones slowly")
axes[1].legend()
plt.tight_layout()
plt.show()

加上位置编码再打乱一次。注意位置编码要加在**打乱之后**的序列上——位置是"第几个座位"，不随 token 搬家：

In [ ]:
pos16 = PositionalEncoding(16).eval()
with torch.no_grad():
    # 在这里写你的代码
    raise NotImplementedError
print(f"有位置编码：max |f(PE(x[perm])) − f(PE(x))[perm]| = {diff_with_pe:.3f}")
ck.permutation_equivariance(diff_tokens, diff_with_pe)

### 搭一个 Transformer 分类器

把积木拼起来：**Embedding → + 位置编码 → N × EncoderBlock → 对 token 取平均（mean-pool）→ Linear 输出类别**。作业里你要用它在 IMDB 上做情感分类，并补上 padding 掩码（讲义这版先不处理 pad）。

In [ ]:
class TransformerClassifier(nn.Module):
    """Embedding + 位置编码 + N 个编码器块 + mean-pool + 线性分类头（讲义版：暂不处理 padding）"""

    def __init__(self, vocab_size, d_model=64, num_heads=2, num_layers=2, d_ff=128,
                 num_classes=2, dropout=0.1, max_len=512):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model, max_len)
        self.blocks = nn.ModuleList([EncoderBlock(d_model, num_heads, d_ff, dropout)
                                     for _ in range(num_layers)])
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x):                        # x: (B, T) 的 token id
        # 在这里写你的代码
        raise NotImplementedError


clf = TransformerClassifier(vocab_size=20000)
fake = torch.randint(1, 20000, (4, 256))         # (B, T)：4 条假影评，各 256 个 token
print("logits:", tuple(clf(fake).shape))
n_params = sum(p.numel() for p in clf.parameters())
print(f"参数量 {n_params:,}，其中 embedding 占 {clf.embed.weight.numel():,}")

---
## S4 · 玩具任务：序列反转

UvA 教程用来检验 Transformer 的原题：输入 16 个 0–9 的随机数字，输出把它**倒过来**：

$$[3, 1, 4, 1, 5, \dots] \;\to\; [\dots, 5, 1, 4, 1, 3]$$

第 $i$ 个位置的答案是第 $L-1-i$ 个输入——**只和位置有关、和内容无关**。没有位置信息的模型注定做不到，正好拿来检验位置编码。设定照 UvA：5 万条训练序列、1 层、1 个头、$d_{\text{model}} = 32$；UvA 用 Lightning 训 10 个 epoch，我们用第 8 周的循环训 3 个。

In [ ]:
def make_reverse_data(n, seq_len=16, num_digits=10, seed=0):
    """n 条随机数字序列及其反转"""
    g = torch.Generator().manual_seed(seed)
    x = torch.randint(num_digits, (n, seq_len), generator=g)   # (N, 16)
    return x, x.flip(1)                                         # 标签：每行倒过来


X_train, Y_train = make_reverse_data(50000, seed=0)
X_test, Y_test = make_reverse_data(10000, seed=1)
print("输入:", X_train[0].tolist())
print("标签:", Y_train[0].tolist())

In [ ]:
class ReversalModel(nn.Module):
    """逐位置分类：Embedding (+ 位置编码) + 1 个 EncoderBlock + Linear(→ 10 类)"""

    def __init__(self, num_digits=10, d_model=32, num_heads=1, d_ff=64, use_pe=True):
        super().__init__()
        self.embed = nn.Embedding(num_digits, d_model)
        self.pos = PositionalEncoding(d_model) if use_pe else nn.Identity()   # 开关：有 / 无位置编码
        self.block = EncoderBlock(d_model, num_heads, d_ff)
        self.out = nn.Linear(d_model, num_digits)

    def forward(self, x):                       # x: (B, T)
        h = self.pos(self.embed(x))             # (B, T, C)
        h = self.block(h)                       # (B, T, C)
        return self.out(h)                      # (B, T, 10)：每个位置做一次 10 类分类


from torch.utils.data import DataLoader, TensorDataset


def train_reversal(use_pe, epochs=3, seed=42):
    """训练并返回 (model, 测试集逐位置准确率)"""
    experiments.set_seed(seed)
    model = ReversalModel(use_pe=use_pe).to(device)
    loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=128, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            # 在这里写你的代码
            raise NotImplementedError
        model.eval()
        with torch.no_grad():
            pred = model(X_test.to(device)).argmax(dim=-1).cpu()        # (N, T)
        acc = (pred == Y_test).float().mean().item()
        print(f"  epoch {epoch + 1}: train loss {loss.item():.4f} | test acc {acc:.4f}")
    return model, acc


t0 = time.perf_counter()
model_pe, acc_pe = train_reversal(use_pe=True)
print(f"有位置编码：测试准确率 {acc_pe:.4f}（{time.perf_counter() - t0:.1f} 秒）")
ck.reversal_accuracy_with_pe(acc_pe)

`nn.CrossEntropyLoss` 要的是 `(N, 类别数)` 的 logits 和 `(N,)` 的标签，所以把 16 个位置拉平：`(B, T, 10)` → `(B*T, 10)`。

模型到底学会了什么？把唯一那个头的注意力权重画出来：

In [ ]:
seq = X_test[:1].to(device)                                          # (1, 16)
with torch.no_grad():
    h = model_pe.pos(model_pe.embed(seq))                             # (1, 16, 32)
    _, attn_rev = model_pe.block.self_attn(h, return_attention=True)  # (1, 1, 16, 16)

fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(attn_rev[0, 0].cpu(), cmap="viridis")
ax.set_xlabel("Key position (input)")
ax.set_ylabel("Query position (output)")
ax.set_title("The head attends to position L-1-i")
plt.show()

一条清清楚楚的反对角线：输出位置 $i$ 把几乎全部注意力放在输入位置 $15 - i$ 上——模型自己"发现"了反转规则。现在把位置编码拿掉：

In [ ]:
t0 = time.perf_counter()
model_nope, acc_nope = train_reversal(use_pe=False)
print(f"无位置编码：测试准确率 {acc_nope:.4f}（{time.perf_counter() - t0:.1f} 秒）")
ck.reversal_accuracy_without_pe(acc_nope)

# 一个置换等变的模型最多能做到什么？位置 i 只知道"自己是几"和"整串里每个数字出现了几次"，
# 而答案是另一个位置上的数——所以最好的猜法是"除了自己以外出现最多的数字"
counts = F.one_hot(X_test, 10).sum(dim=1)                  # (N, 10)：每串里各数字出现几次
mode = counts.argmax(dim=1)                                # (N,)：出现最多的数字
mode_acc = (Y_test == mode[:, None]).float().mean().item()
others = counts[:, None, :] - F.one_hot(X_test, 10)        # (N, 16, 10)：去掉自己之后的计数
best_acc = (Y_test == others.argmax(dim=-1)).float().mean().item()
print(f"'永远猜整串的众数'的准确率: {mode_acc:.4f} | '猜除了自己以外出现最多的数字': {best_acc:.4f}")

无位置编码的模型卡在 0.24 左右，和"猜除了自己以外出现最多的数字"几乎一样（"永远猜整串的众数"约 0.237，略低一点）——它学到的已经是置换等变模型能做到的**最好**策略：不知道自己在哪个位置时，位置 $i$ 只知道"自己是几"和"整串里每个数字出现了几次"，而答案是另一个位置上的数，最好的猜法就是"除了自己以外出现最多的那个"。不是没训好，是结构决定了天花板。

### 本周小结

- `nn.Embedding` = one-hot × 矩阵 = 查表
- 注意力 = 按相似度（$q \cdot k$）加权平均 value：$\mathrm{softmax}(QK^\top/\sqrt{d_k})V$
- 除以 $\sqrt{d_k}$：点积方差是 $d_k$，不缩放 softmax 会饱和、梯度消失
- 因果掩码（下三角）与 padding 掩码（`(B, 1, 1, T)`）：softmax 之前填 $-\infty$
- 多头 = 并行做 $H$ 组注意力，`qkv_proj` → reshape → permute → chunk，每一步写 shape 注释
- 编码器块 = 注意力（交流）+ FFN（思考）+ 残差 + LayerNorm
- 注意力本身置换等变：打乱输入，输出只是被同样打乱；位置编码把"第几个"注入进去。序列反转：有 PE 100%，无 PE 卡在约 0.24 的结构天花板（≈ 猜"除了自己以外出现最多的数字"）

**延伸阅读**：[The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/)（harvardnlp，仓库 [MIT](https://github.com/harvardnlp/annotated-transformer)）——逐段对照原论文实现完整的翻译 Transformer（编码器 + 解码器）。

**预告**：第 12 周用**同一套积木**搭 GPT——只需加上因果掩码和一个"预测下一个字符"的输出头，就能在莎士比亚全集上训练一个会写台词的小语言模型。